In [ ]:
import os, glob
import numpy as np
import nibabel as nib

# ====== 你只需要改这里 ======
DATA_ROOT = "/home/pumengyu/First2TB/PuMengYu/CT/segmentation/Task03_Liver"  # 改成你的liver任务路径
LABEL_DIR = os.path.join(DATA_ROOT, "labelsTr")
TUMOR_ID = 2
# ===========================

def main():
    labs = sorted(glob.glob(os.path.join(LABEL_DIR, "*.nii.gz")))
    if not labs:
        raise RuntimeError(f"No labels found in {LABEL_DIR}")

    total = len(labs)
    has_tumor = 0
    uniq_all = set()
    tumor_vox_list = []
    a=0
    for p in labs:
        a+=1
        
        lab = nib.load(p).get_fdata()
        u = np.unique(lab).astype(int).tolist()
        for x in u:
            uniq_all.add(int(x))

        vox = int(np.sum(lab == TUMOR_ID))
        if vox > 0:
            has_tumor += 1
            tumor_vox_list.append(vox)
        if a%10==0:
            print("a=",a)

    print("=== RAW LABEL CHECK ===")
    print("label_dir:", LABEL_DIR)
    print("total_cases:", total)
    print("unique_labels_overall:", sorted(list(uniq_all)))
    print(f"cases_with_tumor_id_{TUMOR_ID}:", has_tumor, f"({has_tumor/total*100:.2f}%)")

    if tumor_vox_list:
        arr = np.array(tumor_vox_list)
        print("tumor_vox stats (only tumor cases):",
              "min", int(arr.min()),
              "median", int(np.median(arr)),
              "p90", int(np.percentile(arr, 90)),
              "max", int(arr.max()))
    else:
        print("WARNING: no tumor voxels found at all. Either tumor_id is wrong or dataset has no tumor labels.")

if __name__ == "__main__":
    main()

a= 10
=== RAW LABEL CHECK ===
label_dir: /home/pumengyu/First2TB/PuMengYu/CT/segmentation/Task03_Liver/labelsTr
total_cases: 131
unique_labels_overall: [0, 1, 2]
cases_with_tumor_id_2: 118 (90.08%)
tumor_vox stats (only tumor cases): min 98 median 18412 p90 390642 max 1909785


In [ ]:
import os
import torch
from medseg.data.msd import load_msd_dataset, fixed_split
from medseg.data.transforms import build_train_transforms, build_val_transforms
from monai.data import Dataset, DataLoader

DATA_ROOT = "/home/pumengyu/First2TB/PuMengYu/CT/segmentation/Task03_Liver"  # 改
PATCH = (96, 96, 96)
VAL_RATIO = 0.2
SEED = 0
N_PRINT = 20  # 打印多少个样本看看

def inspect(loader, name: str):
    print(f"\n=== {name} TRANSFORMS CHECK ===")
    cnt = 0
    has2 = 0
    for batch in loader:
        y = batch["label"]  # torch tensor after transforms
        if y.ndim == 4:
            y = y.unsqueeze(1)
        y = y.long()
        yy = y[:, 0]

        u = torch.unique(yy).cpu().tolist()
        tumor_vox = int((yy == 2).sum().item())
        liver_vox = int((yy == 1).sum().item())
        flag = tumor_vox > 0
        has2 += int(flag)
        print(f"[{cnt:03d}] unique={u} liver_vox={liver_vox} tumor_vox={tumor_vox} has_tumor={flag}")

        cnt += 1
        if cnt >= N_PRINT:
            break

    print(f"printed {cnt} samples, has_tumor_in_printed = {has2}/{cnt}")

def main():
    items, _ = load_msd_dataset(DATA_ROOT)
    tr, va = fixed_split(items, val_ratio=VAL_RATIO, seed=SEED)

    # 关键：不用 CacheDataset，避免缓存干扰，纯检查
    train_ds = Dataset(tr, transform=build_train_transforms(PATCH))
    val_ds   = Dataset(va, transform=build_val_transforms())

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=False, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

    inspect(train_loader, "TRAIN")
    inspect(val_loader, "VAL")

if __name__ == "__main__":
    main()
# python -m scripts.check_raw_tumor_labels